# 08 · Raporlardan LLM ile etiket çıkarma (Kaggle, 2×T4)

**Neden?** Görüntü modelinin tavanı etiket kalitesidir. Mevcut iki public kaynağın 58 altın
vakadaki AUC'si ~0.89; zayıf bulgular Synovitis (0.79), Fracture (0.79), Lateral OA (0.83),
Contusion (0.86). Test setinde rapor yok, yani metin **yalnızca eğitim hedefi** üretmek için
kullanılabilir — bu notebook tam olarak bunu yapar.

**Yöntem (BTK Datathon'daki rubrik yaklaşımının bu probleme uyarlanmış hâli):**
1. Her rapor, açık tanımlar içeren sabit bir sistem prompt'u ile bir LLM'e okutulur.
2. LLM her bulgu için tek harf döner: **P** (var), **A** (yok / normal), **N** (bahsedilmiyor),
   **U** (belirsiz / şüpheli). JSON şeması sabit, serbest metin yok.
3. Harfler sabit bir tabloyla olasılığa çevrilir (bkz. `src/labels.py`), sonra mevcut
   kaynaklarla ortalanır. Eşleme altın vakalara göre **ayarlanmaz** (58 vaka buna yetmez).
4. Sonuç notebook içinde 58 altın vakaya karşı ölçülür.

**Not (v2):** ilk koşu en uzun raporlarda bellek yetmezliği (OOM) ile 1.464/4.407'de durdu.
Bu sürüm kısmi çıktıyı (`llm_codes_partial*.csv`, `rsna-knee-src` içinde) okuyup kaldığı yerden
devam eder ve batch boyutunu rapor uzunluğuna göre ayarlar.

**Model:** `Qwen/Qwen2.5-7B-Instruct` (çok dilli: raporlar İngilizce, İspanyolca, Fransızca,
Hollandaca, Almanca, Yunanca, Türkçe…). fp16, iki T4'e bölünmüş. Internet: **açık**.

In [ ]:
# Proje kökünü (src/ klasörünü içeren yer) Python yoluna ekle: yerelde ve Kaggle'da çalışır.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk + budama: yarışmanın DICOM klasörlerine girmez (rglob dakikalar sürüyordu)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ bulunamadı")

PROJECT_ROOT = add_project_root()
print("Proje kökü:", PROJECT_ROOT)

In [ ]:
import json, os, re, time
from pathlib import Path
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.config import DATA_DIR, EXPERIMENTS_DIR, IS_KAGGLE, TARGET_COLS

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
BATCH = 24
MAX_NEW = 120
OUT = EXPERIMENTS_DIR
train = pd.read_csv(DATA_DIR / "train.csv")
if not IS_KAGGLE:  # yerel deneme: 3 rapor, küçük model
    MODEL_ID, BATCH = "Qwen/Qwen2.5-0.5B-Instruct", 3
    train = train.head(3)
print(len(train), "rapor |", MODEL_ID, "| GPU:", torch.cuda.device_count())

## 1. Sistem prompt'u: bulgu tanımları

Tanımlar yarışmanın veri açıklamasından alındı ve yaygın belirsizlikler için kurallar eklendi
(ör. "menisküs dejenerasyonu, yırtık değil" → A; "trikompartmanal artroz" → üç OA da P).
Prompt'un her rapor için aynı ve önde olması, tokenizer önbelleği açısından da verimlidir.

In [ ]:
KEYS = ["ACL", "MCL", "MM", "LM", "MOA", "LOA", "PFOA", "EFF", "SYN", "BAKER", "CONT", "FX"]
KEY2COL = dict(zip(KEYS, TARGET_COLS))

SYSTEM = """You are an expert musculoskeletal radiologist. You read one knee MRI report (it may be in any language) and classify twelve findings strictly from what the report says.

For each finding answer with one letter:
P = present (stated as present, including partial/low-grade/small, and post-traumatic or chronic)
A = absent (explicitly normal/intact/no ..., or the structure is described without this abnormality)
N = not mentioned at all
U = uncertain (suspected, possible, cannot exclude, equivocal)

Findings and rules:
ACL  = anterior cruciate ligament injury: sprain, partial or complete tear, rupture, mucoid degeneration with tear, torn graft. Intact graft after reconstruction = A.
MCL  = medial collateral ligament injury: sprain, strain, grade 1-3 injury, partial/complete tear, periligamentous edema of the MCL.
MM   = medial meniscus tear (any tear type, including root tear, flap, bucket-handle, horizontal, complex, re-tear). Degeneration/intrasubstance signal WITHOUT tear = A. Post-meniscectomy without new tear = A.
LM   = lateral meniscus tear, same rules as MM.
MOA  = osteoarthritis of the MEDIAL tibiofemoral compartment: cartilage loss/thinning/chondral defect/chondromalacia of medial femoral condyle or medial tibial plateau, medial osteophytes, medial joint space narrowing, medial compartment arthrosis.
LOA  = same for the LATERAL tibiofemoral compartment.
PFOA = patellofemoral osteoarthritis: chondromalacia patellae (any grade), patellar or trochlear cartilage loss/defect, patellofemoral osteophytes or arthrosis.
       If the report says generalized/tricompartmental/global osteoarthritis or gonarthrosis without sides, mark MOA, LOA and PFOA all P.
EFF  = joint effusion of any size (small, moderate, large, hemarthrosis, lipohemarthrosis). "Physiological/minimal physiological fluid" = A.
SYN  = synovitis: synovitis, synovial thickening/proliferation/hypertrophy, effusion-synovitis, Hoffa fat pad synovitis.
BAKER = Baker's (popliteal) cyst of any size, including ruptured.
CONT = bone contusion / bone bruise / post-traumatic bone marrow edema. Degenerative subchondral edema in osteoarthritis = A unless called contusion/bruise.
FX   = any fracture: avulsion (e.g. Segond), impaction, osteochondral, subchondral insufficiency fracture, stress fracture, tibial plateau fracture, patellar fracture.

Return ONLY a JSON object with exactly these keys and one letter each:
{"ACL":"","MCL":"","MM":"","LM":"","MOA":"","LOA":"","PFOA":"","EFF":"","SYN":"","BAKER":"","CONT":"","FX":""}"""

## 2. Modeli yükle ve toplu (batch) üretim yap

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()

def build_prompt(report: str) -> str:
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": "REPORT:\n" + str(report)[:4000]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

LETTER = re.compile(r'"(\w+)"\s*:\s*"([PANU])"')
def parse(text: str) -> dict:
    found = dict(LETTER.findall(text))
    return {k: found.get(k, "") for k in KEYS}

# Kaldığı yerden devam: önceki koşunun kısmi çıktısı varsa o vakaları atla
done = pd.concat([pd.read_csv(p) for p in Path("/kaggle/input").rglob("llm_codes_partial*.csv")]) if IS_KAGGLE else pd.DataFrame()
done = done.drop_duplicates("StudyInstanceUID") if len(done) else done
rows = done.to_dict("records") if len(done) else []
todo = train[~train.StudyInstanceUID.isin(set(done.StudyInstanceUID) if len(done) else set())]
print("önceden tamamlanan:", len(done), "| kalan:", len(todo))

# Uzunluğa göre sırala (dolgu azalır) ve batch'i token bütçesine göre ayarla:
# ilk koşuda sabit batch=24, en uzun raporlarda GPU belleğini aştı (OOM).
TOKEN_BUDGET = 14000          # batch başına (prompt uzunluğu x batch boyutu) üst sınırı
lens = todo["Report"].fillna("").map(lambda r: len(tok(build_prompt(r)).input_ids))
order = lens.sort_values().index

def run_batch(idx):
    prompts = [build_prompt(r) for r in train.loc[idx, "Report"].fillna("")]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

t0, pos, n_batches = time.time(), 0, 0
order = list(order)
while pos < len(order):
    bs = max(1, min(BATCH, TOKEN_BUDGET // int(lens[order[pos:pos + BATCH]].max())))
    idx = order[pos:pos + bs]
    try:
        texts = run_batch(idx)
    except torch.cuda.OutOfMemoryError:   # yine de sığmazsa batch'i yarıya indir
        torch.cuda.empty_cache()
        BATCH = max(1, bs // 2)
        print("OOM -> batch", BATCH, flush=True)
        continue
    for i, t in zip(idx, texts):
        rows.append({"StudyInstanceUID": train.at[i, "StudyInstanceUID"], "raw": t, **parse(t)})
    pos += len(idx); n_batches += 1
    if n_batches % 10 == 0:
        print(f"{pos}/{len(order)}  bs={bs}  {time.time() - t0:.0f}s  (~{(time.time() - t0) / pos * (len(order) - pos) / 60:.0f} dk kaldı)", flush=True)
        pd.DataFrame(rows).to_csv(OUT / "llm_codes_partial_run.csv", index=False)

codes = pd.DataFrame(rows)
codes.to_csv(OUT / "llm_codes_qwen7b.csv", index=False)
print("Bitti:", codes.shape, f"{(time.time() - t0) / 60:.1f} dk")

## 3. Kalite kontrol

* **Ayrıştırma hatası:** boş kalan harf oranı (model JSON şemasına uymamışsa).
* **Harf dağılımı:** bulgu bazında P/A/N/U oranları.
* **Altın vaka AUC'si:** harfler sabit tabloyla olasılığa çevrilip 58 vakaya karşı ölçülür.

In [ ]:
print("boş (ayrıştırılamayan) oran:", round((codes[KEYS] == "").mean().mean(), 4))
dist = pd.DataFrame({k: codes[k].value_counts(normalize=True) for k in KEYS}).T.fillna(0).round(3)
dist.index = [KEY2COL[k] for k in dist.index]
dist

In [ ]:
from sklearn.metrics import roc_auc_score
CODE_P = {"P": 0.95, "U": 0.6, "N": 0.08, "A": 0.03, "": 0.15}
probs = codes[["StudyInstanceUID"]].copy()
for k in KEYS:
    probs[KEY2COL[k]] = codes[k].map(CODE_P).fillna(0.15)
probs.to_csv(OUT / "llm_labels_qwen7b.csv", index=False)

gold = train[train["ACL"].notna()].set_index("StudyInstanceUID")[TARGET_COLS]
p = probs.set_index("StudyInstanceUID").reindex(gold.index)
aucs = {c: roc_auc_score(gold[c], p[c]) for c in TARGET_COLS if gold[c].nunique() > 1}
print("Altın vaka makro AUC:", round(np.mean(list(aucs.values())), 4))
pd.Series(aucs).round(3).to_frame("AUC").T